# 01. MVTec 전체 category 정밀 학습·평가·시각화

Drive의 MVTec AD 전체 category에 대해 AutoEncoder와 PatchCore를 학습하고 평가합니다.

이번 버전의 핵심 변경사항:
- AutoEncoder를 더 오래 학습하되 validation split + early stopping으로 과적합을 방지합니다.
- train/validation loss history를 CSV/PNG로 저장합니다.
- 완료된 category/model은 건너뛰고, AutoEncoder는 10 epoch마다 Drive에 resume checkpoint를 저장하여 끊긴 지점부터 재개합니다.
- Colab 기본 라이브러리 조합을 유지하며 `requirements.txt` 전체 설치를 하지 않습니다.

산출물:
- category × model별 metrics CSV
- category별 최고 모델 CSV
- 모델/category 성능 비교 그래프
- AutoEncoder 학습 곡선 그래프
- Streamlit 데모용 sample pool metadata
- EC2 배포용 model registry


In [1]:
# ===== 공통 환경 설정: Colab + Drive + GitHub repo + 안전한 import 경로 =====
# 이 셀은 모든 노트북에서 가장 먼저 실행하세요.
# 핵심 원칙:
# - requirements.txt 전체 설치 금지
# - numpy / pandas / torch / torchvision / opencv / scikit-learn 강제 재설치 금지
# - 데이터, checkpoint, deploy bundle은 Google Drive에 저장
# - GitHub에는 코드, README, 작은 시각화 파일, demo sample 이미지만 업로드

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from getpass import getpass

GITHUB_REPO_URL = "https://github.com/wnstjq0915/DefectVision-AD-Proj.git"
GITHUB_USERNAME = "wnstjq0915"
GITHUB_EMAIL = "wnstjq0915@gmail.com"
PROJECT_DIR = Path("/content/DefectVision-AD-Proj")
PROJECT_NAME = "DefectVision-AD"

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception as exc:
    IN_COLAB = False
    print('Colab이 아닌 환경입니다. Drive mount 생략:', repr(exc))

DRIVE_ROOT = Path('/content/drive/MyDrive') / PROJECT_NAME if IN_COLAB else Path.cwd() / 'drive_sim' / PROJECT_NAME
DATA_ROOT = DRIVE_ROOT / 'data' / 'raw'
MVTEC_ROOT = DATA_ROOT / 'mvtec'
VISA_ROOT = DATA_ROOT / 'visa_mvtec'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
RESULT_ROOT = OUTPUT_ROOT / 'multi_category_results'
CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
DEPLOY_ROOT = DRIVE_ROOT / 'deploy'

for p in [DRIVE_ROOT, DATA_ROOT, OUTPUT_ROOT, RESULT_ROOT, CHECKPOINT_ROOT, DEPLOY_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# repo clone 또는 재사용
FORCE_RECLONE = False
if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    print('GitHub repo clone:', GITHUB_REPO_URL)
    subprocess.check_call(['git', 'clone', GITHUB_REPO_URL, str(PROJECT_DIR)])
else:
    print('기존 GitHub repo 사용:', PROJECT_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Git 사용자 정보는 commit용. push 인증은 별도 token 환경변수 사용.
subprocess.run(['git', 'config', '--global', 'user.email', GITHUB_EMAIL], check=False)
subprocess.run(['git', 'config', '--global', 'user.name', GITHUB_USERNAME], check=False)

print('IN_COLAB       =', IN_COLAB)
print('PROJECT_DIR    =', PROJECT_DIR)
print('DRIVE_ROOT     =', DRIVE_ROOT)
print('MVTEC_ROOT     =', MVTEC_ROOT)
print('VISA_ROOT      =', VISA_ROOT)
print('RESULT_ROOT    =', RESULT_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

# Colab 기본 패키지 버전 확인. 버전 꼬임 방지를 위해 강제 설치하지 않음.
import importlib
print('\n===== 주요 패키지 버전 =====')
for name in ['numpy', 'pandas', 'torch', 'torchvision', 'cv2', 'sklearn', 'matplotlib', 'PIL', 'yaml', 'tqdm']:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'unknown')
        if name == 'PIL':
            from PIL import Image
            ver = Image.__version__
        print(f'{name:12s}: {ver}')
    except Exception as exc:
        print(f'{name:12s}: IMPORT ERROR -> {repr(exc)}')

# 작은 유틸 패키지만 누락 시 설치. 핵심 ML 패키지는 설치하지 않음.
for import_name, pip_name in [('yaml', 'PyYAML'), ('tqdm', 'tqdm')]:
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

# torch.load weights_only 기본값 변경에 대비한 안전 패치
# PyTorch 2.6+에서는 torch.load 기본 weights_only가 바뀌어 기존 checkpoint 로드가 실패할 수 있음.
inference_path = PROJECT_DIR / 'src' / 'inference.py'
if inference_path.exists():
    text = inference_path.read_text(encoding='utf-8')
    old = 'checkpoint = torch.load(checkpoint_path, map_location=resolved_device)'
    new = """\n    try:\n        checkpoint = torch.load(checkpoint_path, map_location=resolved_device, weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(checkpoint_path, map_location=resolved_device)\n    """.rstrip()
    if old in text:
        text = text.replace(old, new)
        inference_path.write_text(text, encoding='utf-8')
        print('patched:', inference_path)

print('\n현재 작업 디렉토리:', Path.cwd())
print('src 존재 여부:', (PROJECT_DIR / 'src').exists())

Mounted at /content/drive
GitHub repo clone: https://github.com/wnstjq0915/DefectVision-AD-Proj.git
IN_COLAB       = True
PROJECT_DIR    = /content/DefectVision-AD-Proj
DRIVE_ROOT     = /content/drive/MyDrive/DefectVision-AD
MVTEC_ROOT     = /content/drive/MyDrive/DefectVision-AD/data/raw/mvtec
VISA_ROOT      = /content/drive/MyDrive/DefectVision-AD/data/raw/visa_mvtec
RESULT_ROOT    = /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results
CHECKPOINT_ROOT= /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints

===== 주요 패키지 버전 =====
numpy       : 2.0.2
pandas      : 2.2.2
torch       : 2.11.0+cpu
torchvision : 0.26.0+cpu
cv2         : 4.13.0
sklearn     : 1.6.1
matplotlib  : 3.10.0
PIL         : 11.3.0
yaml        : 6.0.3
tqdm        : 4.67.3
patched: /content/DefectVision-AD-Proj/src/inference.py

현재 작업 디렉토리: /content/DefectVision-AD-Proj
src 존재 여부: True


In [2]:
# ===== Dataset discovery / manifest helper =====
from pathlib import Path
import csv
import json
import random
from collections import Counter, defaultdict
from typing import Iterable

IMAGE_EXTENSIONS = {'.bmp', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}


def image_files(directory: Path):
    if not directory.exists():
        return []
    return sorted(p for p in directory.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def discover_mvtec_categories(root: Path):
    root = Path(root)
    if not root.exists():
        return []
    cats = []
    for p in sorted(root.iterdir()):
        if not p.is_dir() or p.name.startswith('.'):
            continue
        if (p / 'train' / 'good').exists() and (p / 'test').exists():
            cats.append(p.name)
    return cats


def summarize_mvtec_category(root: Path, category: str):
    cat_root = Path(root) / category
    row = {
        'dataset': 'mvtec',
        'category': category,
        'category_root': str(cat_root),
        'train_good': 0,
        'test_good': 0,
        'test_anomaly': 0,
        'test_total': 0,
        'defect_types': '',
        'has_ground_truth': False,
        'status': 'missing',
    }
    if not cat_root.exists():
        return row
    train_good = image_files(cat_root / 'train' / 'good')
    test_counts = {}
    test_root = cat_root / 'test'
    if test_root.exists():
        for d in sorted(x for x in test_root.iterdir() if x.is_dir()):
            test_counts[d.name] = len(image_files(d))
    row['train_good'] = len(train_good)
    row['test_good'] = test_counts.get('good', 0)
    row['test_anomaly'] = sum(v for k, v in test_counts.items() if k != 'good')
    row['test_total'] = sum(test_counts.values())
    row['defect_types'] = ', '.join(k for k in sorted(test_counts) if k != 'good')
    row['has_ground_truth'] = (cat_root / 'ground_truth').exists()
    row['status'] = 'ok' if row['train_good'] and row['test_total'] else 'incomplete'
    return row


def write_csv(rows, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8')
        return path
    fieldnames = list(rows[0].keys())
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

In [3]:
# ===== 실험 파라미터: 정밀 AutoEncoder 학습 + 과적합 방지 + 중단 재개 =====
import math
import time
import yaml
import numpy as np

# 빠른 흐름 확인만 할 때 True로 바꾸세요.
RUN_SMOKE_TEST = False

RUN_AUTOENCODER = True
RUN_PATCHCORE = True

# 완료/재개 정책
# - 완료된 category/model은 건너뜁니다.
# - AutoEncoder는 10 epoch마다 Drive에 resume checkpoint를 저장합니다.
# - 런타임이 끊기면 마지막 resume checkpoint 다음 epoch부터 이어서 학습합니다.
SKIP_COMPLETED_RUNS = True
SKIP_EXISTING_CHECKPOINTS = True
FORCE_RETRAIN_AUTOENCODER = False
FORCE_RETRAIN_PATCHCORE = False

AUTOENCODER_RESUME_ENABLED = True
AUTOENCODER_SAVE_EVERY_EPOCHS = 10
AUTOENCODER_KEEP_RESUME_AFTER_SUCCESS = True

IMAGE_SIZE = 256

# Colab/Jupyter 환경에서는 multiprocessing worker 경고와 런타임 불안정을 줄이기 위해 0 권장
NUM_WORKERS = 0

THRESHOLD_PERCENTILE = 95

if RUN_SMOKE_TEST:
    # 빠른 테스트용: 1~2개 category만 짧게 실행
    AUTOENCODER_MAX_EPOCHS = 3
    AUTOENCODER_MIN_EPOCHS = 1
    AUTOENCODER_EARLY_STOPPING_PATIENCE = 2
    AUTOENCODER_BATCH_SIZE = 8
    MAX_CATEGORIES = 2
    PATCHCORE_BATCH_SIZE = 4
    PATCHCORE_MAX_MEMORY_PATCHES = 5000
else:
    # 정식 실행용: validation loss가 개선되는 동안 충분히 학습
    AUTOENCODER_MAX_EPOCHS = 80
    AUTOENCODER_MIN_EPOCHS = 20
    AUTOENCODER_EARLY_STOPPING_PATIENCE = 4
    AUTOENCODER_BATCH_SIZE = 8
    MAX_CATEGORIES = None
    PATCHCORE_BATCH_SIZE = 4
    PATCHCORE_MAX_MEMORY_PATCHES = 50000

# AutoEncoder 과적합 방지 관련 설정
AUTOENCODER_VAL_RATIO = 0.15
AUTOENCODER_LEARNING_RATE = 5e-4
AUTOENCODER_WEIGHT_DECAY = 1e-4
AUTOENCODER_GRAD_CLIP_NORM = 1.0
AUTOENCODER_EARLY_STOPPING_MIN_DELTA = 1e-5
AUTOENCODER_LR_SCHEDULER_FACTOR = 0.5
AUTOENCODER_LR_SCHEDULER_PATIENCE = 4
AUTOENCODER_AUGMENT = True
AUTOENCODER_USE_AMP = True

AUTOENCODER_BASE_CHANNELS = 32
AUTOENCODER_LATENT_CHANNELS = 256
PATCHCORE_PRETRAINED = True
SAMPLE_POOL_PER_CATEGORY = 30

MODELS_TO_RUN = []
if RUN_AUTOENCODER:
    MODELS_TO_RUN.append('autoencoder')
if RUN_PATCHCORE:
    MODELS_TO_RUN.append('patchcore')

SELECTION_WEIGHTS = {
    'auroc': 0.40,
    'f1': 0.25,
    'pixel_auroc': 0.25,
    'accuracy': 0.10,
}

categories = discover_mvtec_categories(MVTEC_ROOT)
if MAX_CATEGORIES is not None:
    categories = categories[:MAX_CATEGORIES]

print('RUN_SMOKE_TEST:', RUN_SMOKE_TEST)
print('MODELS_TO_RUN:', MODELS_TO_RUN)
print('categories:', len(categories), categories)
print('RESULT_ROOT:', RESULT_ROOT)
print('CHECKPOINT_ROOT:', CHECKPOINT_ROOT)
print()
print('AutoEncoder 정밀 학습 설정')
print(' - max_epochs:', AUTOENCODER_MAX_EPOCHS)
print(' - min_epochs:', AUTOENCODER_MIN_EPOCHS)
print(' - patience:', AUTOENCODER_EARLY_STOPPING_PATIENCE)
print(' - val_ratio:', AUTOENCODER_VAL_RATIO)
print(' - lr:', AUTOENCODER_LEARNING_RATE)
print(' - weight_decay:', AUTOENCODER_WEIGHT_DECAY)
print(' - augment:', AUTOENCODER_AUGMENT)
print(' - skip_completed_runs:', SKIP_COMPLETED_RUNS)
print(' - force_retrain_autoencoder:', FORCE_RETRAIN_AUTOENCODER)
print(' - resume_enabled:', AUTOENCODER_RESUME_ENABLED)
print(' - save_every_epochs:', AUTOENCODER_SAVE_EVERY_EPOCHS)
print(' - num_workers:', NUM_WORKERS)


RUN_SMOKE_TEST: False
MODELS_TO_RUN: ['autoencoder', 'patchcore']
categories: 15 ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
RESULT_ROOT: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results
CHECKPOINT_ROOT: /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints

AutoEncoder 정밀 학습 설정
 - max_epochs: 80
 - min_epochs: 20
 - patience: 4
 - val_ratio: 0.15
 - lr: 0.0005
 - weight_decay: 0.0001
 - augment: True
 - skip_completed_runs: True
 - force_retrain_autoencoder: False
 - resume_enabled: True
 - save_every_epochs: 10
 - num_workers: 0


In [4]:
# ===== 프로젝트 모듈 import =====
from src.datasets import build_dataset
from src.train import set_seed, get_device, train_patchcore
from src.evaluate import evaluate
from src.inference import load_predictor
from src.visualize import save_prediction_panel
from src.config import ensure_parent
from src.models import ConvAutoEncoder, reconstruction_error_map

import copy
import csv
import json
from contextlib import nullcontext
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

set_seed(42)
device = get_device('auto')
print('device:', device)

device: cpu


In [5]:
# ===== AutoEncoder 정밀 학습 함수: validation split + early stopping + LR scheduler + resume checkpoint =====
# src/train.py의 기본 train_autoencoder 대신 이 노트북 안의 함수를 사용합니다.
# checkpoint 형식은 기존 src.inference.load_predictor와 호환되도록 유지합니다.
#
# 추가 기능:
# - 10 epoch마다 Drive에 *_resume.pt 임시 checkpoint 저장
# - 런타임이 끊기면 *_resume.pt에서 optimizer/scheduler/history까지 복원 후 이어서 학습
# - 최종 checkpoint가 이미 있으면 상위 학습 loop에서 해당 category/model을 건너뜀


def clone_config_with_augment(config, augment: bool):
    cloned = copy.deepcopy(config)
    cloned.setdefault('preprocessing', {})['augment'] = bool(augment)
    return cloned


def split_train_val_indices(n_items: int, val_ratio: float, seed: int = 42):
    indices = list(range(n_items))
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)

    if n_items < 3:
        return indices, indices

    val_size = int(round(n_items * val_ratio))
    val_size = max(1, min(val_size, n_items - 1))
    val_indices = sorted(indices[:val_size])
    train_indices = sorted(indices[val_size:])
    return train_indices, val_indices


def make_autoencoder_loaders(config):
    dataset_config = config.get('dataset', {})
    train_config = config.get('train', {})
    batch_size = int(dataset_config.get('batch_size', AUTOENCODER_BATCH_SIZE))
    num_workers = int(dataset_config.get('num_workers', NUM_WORKERS))
    val_ratio = float(train_config.get('val_ratio', AUTOENCODER_VAL_RATIO))
    seed = int(config.get('experiment', {}).get('seed', 42))

    # train dataset은 augmentation 적용
    train_aug_config = clone_config_with_augment(config, bool(config.get('preprocessing', {}).get('augment', False)))
    train_dataset_aug = build_dataset(train_aug_config, split='train')

    # validation/threshold dataset은 augmentation 미적용
    clean_config = clone_config_with_augment(config, False)
    train_dataset_clean = build_dataset(clean_config, split='train')

    train_indices, val_indices = split_train_val_indices(len(train_dataset_clean), val_ratio, seed)

    train_subset = Subset(train_dataset_aug, train_indices)
    val_subset = Subset(train_dataset_clean, val_indices)

    loader_kwargs = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    # num_workers=0일 때 persistent_workers를 넣으면 PyTorch에서 오류가 날 수 있습니다.
    if num_workers > 0:
        loader_kwargs['persistent_workers'] = False

    train_loader = DataLoader(
        train_subset,
        shuffle=True,
        drop_last=False,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        val_subset,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )
    threshold_loader = DataLoader(
        train_dataset_clean,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )

    return train_loader, val_loader, threshold_loader, train_indices, val_indices


def amp_autocast_context(device, enabled: bool):
    if not enabled or device.type != 'cuda':
        return nullcontext()
    try:
        return torch.amp.autocast(device_type='cuda')
    except Exception:
        return torch.cuda.amp.autocast()


def make_grad_scaler(device, enabled: bool):
    if not enabled or device.type != 'cuda':
        return None
    try:
        return torch.amp.GradScaler('cuda', enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)


@torch.no_grad()
def autoencoder_epoch_loss(model, loader, device, criterion):
    model.eval()
    running = 0.0
    n = 0
    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        reconstructions = model(images)
        loss = criterion(reconstructions, images)
        running += float(loss.item()) * images.size(0)
        n += images.size(0)
    return running / max(n, 1)


@torch.no_grad()
def autoencoder_image_scores(model, dataloader, device):
    model.eval()
    scores = []
    for batch in tqdm(dataloader, desc='threshold scores', leave=False):
        images = batch['image'].to(device, non_blocking=True)
        reconstructions = model(images)
        error_map = reconstruction_error_map(images, reconstructions)
        scores.extend(error_map.flatten(1).mean(dim=1).detach().cpu().tolist())
    return scores


def cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def save_autoencoder_history(history, history_csv_path, history_png_path, title):
    history_csv_path = Path(history_csv_path)
    history_png_path = Path(history_png_path)
    history_csv_path.parent.mkdir(parents=True, exist_ok=True)
    history_png_path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = ['epoch', 'train_loss', 'val_loss', 'best_val_loss', 'learning_rate', 'no_improve', 'is_best']
    with history_csv_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(history)

    if history:
        epochs = [row['epoch'] for row in history]
        train_losses = [row['train_loss'] for row in history]
        val_losses = [row['val_loss'] for row in history]

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(epochs, train_losses, marker='o', linewidth=1.5, label='train loss')
        ax.plot(epochs, val_losses, marker='o', linewidth=1.5, label='validation loss')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE reconstruction loss')
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(history_png_path, dpi=160)
        plt.close(fig)

    print('saved history csv:', history_csv_path)
    print('saved history png:', history_png_path)


def save_autoencoder_resume_checkpoint(
    resume_checkpoint_path,
    *,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    best_state,
    best_val_loss,
    best_epoch,
    no_improve,
    history,
    config,
    train_indices,
    val_indices,
    history_csv,
    history_png,
):
    """Drive에 중간 재개용 checkpoint를 저장합니다."""
    resume_checkpoint_path = Path(resume_checkpoint_path)
    resume_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    scaler_state = None
    if scaler is not None:
        try:
            scaler_state = scaler.state_dict()
        except Exception:
            scaler_state = None

    payload = {
        'checkpoint_kind': 'autoencoder_resume',
        'epoch': int(epoch),
        'model_type': 'autoencoder',
        'model_state': cpu_state_dict(model),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state': scaler_state,
        'best_state': best_state,
        'best_val_loss': float(best_val_loss) if best_val_loss != float('inf') else float('inf'),
        'best_epoch': int(best_epoch),
        'no_improve': int(no_improve),
        'history': list(history),
        'config': config,
        'train_indices': train_indices,
        'val_indices': val_indices,
        'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    torch.save(payload, resume_checkpoint_path)

    title = f"AutoEncoder loss - {config.get('dataset', {}).get('category', 'unknown')}"
    save_autoencoder_history(history, history_csv, history_png, title)
    print(f'[resume saved] epoch={epoch} path={resume_checkpoint_path}')


def load_autoencoder_resume_checkpoint(resume_checkpoint_path, model, optimizer, scheduler, scaler, device):
    """중간 checkpoint가 있으면 학습 상태를 복원합니다."""
    resume_checkpoint_path = Path(resume_checkpoint_path)
    if not resume_checkpoint_path.exists():
        return None

    print('[resume found]', resume_checkpoint_path)
    try:
        try:
            checkpoint = torch.load(resume_checkpoint_path, map_location=device, weights_only=False)
        except TypeError:
            checkpoint = torch.load(resume_checkpoint_path, map_location=device)

        if checkpoint.get('checkpoint_kind') != 'autoencoder_resume':
            print('[WARN] resume checkpoint 형식이 예상과 다릅니다. 새로 학습합니다.')
            return None

        model.load_state_dict(checkpoint['model_state'])

        if checkpoint.get('optimizer_state') is not None:
            optimizer.load_state_dict(checkpoint['optimizer_state'])

        if scheduler is not None and checkpoint.get('scheduler_state') is not None:
            scheduler.load_state_dict(checkpoint['scheduler_state'])

        if scaler is not None and checkpoint.get('scaler_state') is not None:
            try:
                scaler.load_state_dict(checkpoint['scaler_state'])
            except Exception as exc:
                print('[WARN] scaler state 복원 실패. scaler만 새로 시작합니다:', repr(exc))

        print(f"[resume loaded] last_epoch={checkpoint.get('epoch')} best_epoch={checkpoint.get('best_epoch')} best_val={checkpoint.get('best_val_loss')}")
        return checkpoint

    except Exception as exc:
        print('[WARN] resume checkpoint load failed. 새로 학습합니다:', repr(exc))
        return None


def train_autoencoder_regularized(config, device):
    """AutoEncoder를 validation 기반 early stopping으로 학습합니다.

    - train split의 정상 이미지 중 일부를 validation으로 분리합니다.
    - train subset에만 flip augmentation을 적용합니다.
    - validation loss가 개선된 best checkpoint만 최종 checkpoint로 저장합니다.
    - 10 epoch마다 Drive에 resume checkpoint를 저장합니다.
    - runtime이 끊기면 resume checkpoint 다음 epoch부터 이어서 학습합니다.
    - threshold는 augmentation 없는 전체 train 정상 이미지 score의 percentile로 계산합니다.
    """
    set_seed(int(config.get('experiment', {}).get('seed', 42)))

    train_loader, val_loader, threshold_loader, train_indices, val_indices = make_autoencoder_loaders(config)

    model_config = config.get('model', {})
    train_config = config.get('train', {})
    output_config = config.get('outputs', {})

    checkpoint_path = ensure_parent(output_config.get('checkpoint', 'outputs/checkpoints/autoencoder.pt'))
    checkpoint_path = Path(checkpoint_path)

    history_csv = Path(output_config.get('history_csv', str(checkpoint_path).replace('.pt', '_history.csv')))
    history_png = Path(output_config.get('history_png', str(checkpoint_path).replace('.pt', '_loss.png')))
    resume_checkpoint_path = Path(output_config.get('resume_checkpoint', str(checkpoint_path).replace('.pt', '_resume.pt')))

    model = ConvAutoEncoder(
        in_channels=int(model_config.get('in_channels', 3)),
        base_channels=int(model_config.get('base_channels', 32)),
        latent_channels=int(model_config.get('latent_channels', 256)),
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(train_config.get('learning_rate', AUTOENCODER_LEARNING_RATE)),
        weight_decay=float(train_config.get('weight_decay', AUTOENCODER_WEIGHT_DECAY)),
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=float(train_config.get('lr_scheduler_factor', AUTOENCODER_LR_SCHEDULER_FACTOR)),
        patience=int(train_config.get('lr_scheduler_patience', AUTOENCODER_LR_SCHEDULER_PATIENCE)),
    )

    criterion = nn.MSELoss()
    max_epochs = int(train_config.get('epochs', AUTOENCODER_MAX_EPOCHS))
    min_epochs = int(train_config.get('min_epochs', AUTOENCODER_MIN_EPOCHS))
    patience = int(train_config.get('early_stopping_patience', AUTOENCODER_EARLY_STOPPING_PATIENCE))
    min_delta = float(train_config.get('early_stopping_min_delta', AUTOENCODER_EARLY_STOPPING_MIN_DELTA))
    grad_clip_norm = float(train_config.get('grad_clip_norm', AUTOENCODER_GRAD_CLIP_NORM))
    use_amp = bool(train_config.get('use_amp', AUTOENCODER_USE_AMP))
    resume_enabled = bool(train_config.get('resume_enabled', AUTOENCODER_RESUME_ENABLED))
    save_every_epochs = int(train_config.get('save_every_epochs', AUTOENCODER_SAVE_EVERY_EPOCHS))
    keep_resume_after_success = bool(train_config.get('keep_resume_after_success', AUTOENCODER_KEEP_RESUME_AFTER_SUCCESS))

    scaler = make_grad_scaler(device, use_amp)

    best_val_loss = float('inf')
    best_epoch = 0
    best_state = None
    no_improve = 0
    history = []
    start_epoch = 1
    resumed_from_epoch = 0
    started = time.time()

    if resume_enabled:
        resume_payload = load_autoencoder_resume_checkpoint(
            resume_checkpoint_path,
            model,
            optimizer,
            scheduler,
            scaler,
            device,
        )
        if resume_payload is not None:
            resumed_from_epoch = int(resume_payload.get('epoch', 0))
            start_epoch = resumed_from_epoch + 1
            best_val_loss = float(resume_payload.get('best_val_loss', float('inf')))
            best_epoch = int(resume_payload.get('best_epoch', 0))
            best_state = resume_payload.get('best_state')
            no_improve = int(resume_payload.get('no_improve', 0))
            history = list(resume_payload.get('history', []))

    print('AutoEncoder detailed training')
    print(' - train samples:', len(train_loader.dataset))
    print(' - validation samples:', len(val_loader.dataset))
    print(' - max_epochs:', max_epochs, 'min_epochs:', min_epochs, 'patience:', patience)
    print(' - resume_checkpoint:', resume_checkpoint_path)
    if resumed_from_epoch:
        print(f' - resume: epoch {resumed_from_epoch}까지 완료된 상태에서 epoch {start_epoch}부터 재개')
    else:
        print(' - resume: 새 학습 시작')

    if start_epoch > max_epochs:
        print(f'[resume complete] resume checkpoint가 max_epochs={max_epochs}까지 이미 진행되어 있습니다.')
    else:
        try:
            for epoch in range(start_epoch, max_epochs + 1):
                model.train()
                running_loss = 0.0
                n_samples = 0
                progress = tqdm(train_loader, desc=f'epoch {epoch}/{max_epochs}', leave=False)

                for batch in progress:
                    images = batch['image'].to(device, non_blocking=True)
                    optimizer.zero_grad(set_to_none=True)

                    with amp_autocast_context(device, use_amp):
                        reconstructions = model(images)
                        loss = criterion(reconstructions, images)

                    if scaler is not None:
                        scaler.scale(loss).backward()
                        if grad_clip_norm > 0:
                            scaler.unscale_(optimizer)
                            nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        if grad_clip_norm > 0:
                            nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                        optimizer.step()

                    running_loss += float(loss.item()) * images.size(0)
                    n_samples += images.size(0)
                    progress.set_postfix(loss=f'{loss.item():.5f}')

                train_loss = running_loss / max(n_samples, 1)
                val_loss = autoencoder_epoch_loss(model, val_loader, device, criterion)
                scheduler.step(val_loss)
                current_lr = float(optimizer.param_groups[0]['lr'])

                improved = val_loss < (best_val_loss - min_delta)
                if improved:
                    best_val_loss = val_loss
                    best_epoch = epoch
                    best_state = cpu_state_dict(model)
                    no_improve = 0
                else:
                    no_improve += 1

                history.append({
                    'epoch': epoch,
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                    'best_val_loss': best_val_loss,
                    'learning_rate': current_lr,
                    'no_improve': no_improve,
                    'is_best': int(improved),
                })

                mark = '*' if improved else ''
                print(
                    f'epoch={epoch:03d} train_loss={train_loss:.6f} '
                    f'val_loss={val_loss:.6f} best_val={best_val_loss:.6f} '
                    f'lr={current_lr:.2e} no_improve={no_improve}/{patience} {mark}'
                )

                # 사용자가 요청한 Drive 임시저장: 10 epoch마다 저장
                if resume_enabled and save_every_epochs > 0 and epoch % save_every_epochs == 0:
                    save_autoencoder_resume_checkpoint(
                        resume_checkpoint_path,
                        epoch=epoch,
                        model=model,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        scaler=scaler,
                        best_state=best_state,
                        best_val_loss=best_val_loss,
                        best_epoch=best_epoch,
                        no_improve=no_improve,
                        history=history,
                        config=config,
                        train_indices=train_indices,
                        val_indices=val_indices,
                        history_csv=history_csv,
                        history_png=history_png,
                    )

                if epoch >= min_epochs and no_improve >= patience:
                    print(f'early stopping: epoch={epoch}, best_epoch={best_epoch}, best_val_loss={best_val_loss:.6f}')
                    break

        except KeyboardInterrupt:
            print('[INTERRUPTED] 사용자가 실행을 중단했습니다. 마지막 완료 epoch 기준 resume checkpoint를 저장합니다.')
            if resume_enabled and history:
                last_epoch = int(history[-1]['epoch'])
                save_autoencoder_resume_checkpoint(
                    resume_checkpoint_path,
                    epoch=last_epoch,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    best_state=best_state,
                    best_val_loss=best_val_loss,
                    best_epoch=best_epoch,
                    no_improve=no_improve,
                    history=history,
                    config=config,
                    train_indices=train_indices,
                    val_indices=val_indices,
                    history_csv=history_csv,
                    history_png=history_png,
                )
            raise

        except Exception:
            print('[ERROR] 학습 중 오류가 발생했습니다. 마지막 완료 epoch 기준 resume checkpoint를 저장한 뒤 오류를 다시 발생시킵니다.')
            if resume_enabled and history:
                last_epoch = int(history[-1]['epoch'])
                save_autoencoder_resume_checkpoint(
                    resume_checkpoint_path,
                    epoch=last_epoch,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    best_state=best_state,
                    best_val_loss=best_val_loss,
                    best_epoch=best_epoch,
                    no_improve=no_improve,
                    history=history,
                    config=config,
                    train_indices=train_indices,
                    val_indices=val_indices,
                    history_csv=history_csv,
                    history_png=history_png,
                )
            raise

    if best_state is None:
        best_state = cpu_state_dict(model)
        best_epoch = history[-1]['epoch'] if history else 0
        best_val_loss = history[-1]['val_loss'] if history else float('nan')

    # 최종 직전에도 resume checkpoint를 한 번 더 저장합니다.
    if resume_enabled and history:
        last_epoch = int(history[-1]['epoch'])
        save_autoencoder_resume_checkpoint(
            resume_checkpoint_path,
            epoch=last_epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            best_state=best_state,
            best_val_loss=best_val_loss,
            best_epoch=best_epoch,
            no_improve=no_improve,
            history=history,
            config=config,
            train_indices=train_indices,
            val_indices=val_indices,
            history_csv=history_csv,
            history_png=history_png,
        )

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    train_scores = autoencoder_image_scores(model, threshold_loader, device)
    percentile = float(train_config.get('threshold_percentile', THRESHOLD_PERCENTILE))
    threshold = float(np.percentile(train_scores, percentile))

    elapsed_sec = time.time() - started
    final_epoch = history[-1]['epoch'] if history else 0
    training_info = {
        'training_mode': 'regularized_early_stopping_resume',
        'max_epochs': max_epochs,
        'final_epoch': final_epoch,
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'early_stopped': bool(history and final_epoch < max_epochs and no_improve >= patience),
        'elapsed_sec': elapsed_sec,
        'train_size': len(train_loader.dataset),
        'val_size': len(val_loader.dataset),
        'val_ratio': float(train_config.get('val_ratio', AUTOENCODER_VAL_RATIO)),
        'augment': bool(config.get('preprocessing', {}).get('augment', False)),
        'learning_rate': float(train_config.get('learning_rate', AUTOENCODER_LEARNING_RATE)),
        'weight_decay': float(train_config.get('weight_decay', AUTOENCODER_WEIGHT_DECAY)),
        'history_csv': str(history_csv),
        'history_png': str(history_png),
        'resume_enabled': resume_enabled,
        'resume_checkpoint': str(resume_checkpoint_path),
        'resumed_from_epoch': resumed_from_epoch,
        'save_every_epochs': save_every_epochs,
    }

    torch.save(
        {
            'model_type': 'autoencoder',
            'model_state': best_state,
            'threshold': threshold,
            'train_scores': train_scores,
            'config': config,
            'training_info': training_info,
            'training_history': history,
            'train_indices': train_indices,
            'val_indices': val_indices,
        },
        checkpoint_path,
    )

    title = f"AutoEncoder loss - {config.get('dataset', {}).get('category', 'unknown')}"
    save_autoencoder_history(history, history_csv, history_png, title)

    if resume_checkpoint_path.exists() and not keep_resume_after_success:
        try:
            resume_checkpoint_path.unlink()
            print('removed resume checkpoint:', resume_checkpoint_path)
        except Exception as exc:
            print('[WARN] resume checkpoint 삭제 실패:', repr(exc))

    print(f'saved_checkpoint={checkpoint_path}')
    print(f'threshold_p{percentile:g}={threshold:.8f}')
    print('training_info:', json.dumps(training_info, indent=2, ensure_ascii=False))
    return checkpoint_path


def load_autoencoder_training_info(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return {}
    try:
        try:
            checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        except TypeError:
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
        return checkpoint.get('training_info', {}) or {}
    except Exception as exc:
        print('[WARN] training_info load failed:', checkpoint_path, repr(exc))
        return {}


In [6]:
# ===== config / metrics helper =====
import csv
from pathlib import Path


def make_config(category: str, model_name: str):
    batch_size = AUTOENCODER_BATCH_SIZE if model_name == 'autoencoder' else PATCHCORE_BATCH_SIZE
    checkpoint_path = CHECKPOINT_ROOT / 'mvtec' / category / f'{model_name}.pt'
    resume_checkpoint_path = CHECKPOINT_ROOT / 'mvtec' / category / f'{model_name}_resume.pt'
    history_dir = RESULT_ROOT / 'training_history' / 'mvtec' / category

    config = {
        'experiment': {
            'name': f'mvtec_{category}_{model_name}',
            'seed': 42,
            'device': 'auto',
        },
        'dataset': {
            'name': 'mvtec',
            'root': str(MVTEC_ROOT),
            'category': category,
            'image_size': IMAGE_SIZE,
            'batch_size': batch_size,
            'num_workers': NUM_WORKERS,
        },
        'preprocessing': {
            'grayscale': False,
            'gaussian_blur': False,
            'canny': False,
            'hist_equalize': False,
            'augment': model_name == 'autoencoder' and AUTOENCODER_AUGMENT,
            'normalize_mean': [0.485, 0.456, 0.406],
            'normalize_std': [0.229, 0.224, 0.225],
        },
        'train': {
            'threshold_percentile': THRESHOLD_PERCENTILE,
        },
        'outputs': {
            'checkpoint': str(checkpoint_path),
        },
    }

    if model_name == 'autoencoder':
        config['model'] = {
            'type': 'autoencoder',
            'in_channels': 3,
            'base_channels': AUTOENCODER_BASE_CHANNELS,
            'latent_channels': AUTOENCODER_LATENT_CHANNELS,
        }
        config['train'].update({
            'epochs': AUTOENCODER_MAX_EPOCHS,
            'min_epochs': AUTOENCODER_MIN_EPOCHS,
            'learning_rate': AUTOENCODER_LEARNING_RATE,
            'weight_decay': AUTOENCODER_WEIGHT_DECAY,
            'val_ratio': AUTOENCODER_VAL_RATIO,
            'early_stopping_patience': AUTOENCODER_EARLY_STOPPING_PATIENCE,
            'early_stopping_min_delta': AUTOENCODER_EARLY_STOPPING_MIN_DELTA,
            'lr_scheduler_factor': AUTOENCODER_LR_SCHEDULER_FACTOR,
            'lr_scheduler_patience': AUTOENCODER_LR_SCHEDULER_PATIENCE,
            'grad_clip_norm': AUTOENCODER_GRAD_CLIP_NORM,
            'use_amp': AUTOENCODER_USE_AMP,
            'resume_enabled': AUTOENCODER_RESUME_ENABLED,
            'save_every_epochs': AUTOENCODER_SAVE_EVERY_EPOCHS,
            'keep_resume_after_success': AUTOENCODER_KEEP_RESUME_AFTER_SUCCESS,
        })
        config['outputs'].update({
            'history_csv': str(history_dir / 'autoencoder_history.csv'),
            'history_png': str(history_dir / 'autoencoder_loss.png'),
            'resume_checkpoint': str(resume_checkpoint_path),
        })

    elif model_name == 'patchcore':
        config['model'] = {
            'type': 'patchcore',
            'backbone': 'resnet18',
            'pretrained': PATCHCORE_PRETRAINED,
            'max_memory_patches': PATCHCORE_MAX_MEMORY_PATCHES,
        }
    else:
        raise ValueError(model_name)

    return config


def write_yaml(config, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)
    return path


def optional_float(value):
    if value is None:
        return None
    try:
        value = float(value)
        if math.isnan(value):
            return None
        return value
    except Exception:
        return None


def selection_score(metrics, weights=SELECTION_WEIGHTS):
    numerator = 0.0
    denominator = 0.0
    for name, weight in weights.items():
        value = optional_float(metrics.get(name))
        if value is not None:
            numerator += value * weight
            denominator += weight
    return numerator / denominator if denominator else None


def flatten_result(category, model_name, result, config_path, checkpoint_path, status='ok', error=''):
    metrics = result.get('metrics', {}) if isinstance(result, dict) else {}
    row = {
        'dataset': 'mvtec',
        'category': category,
        'model': model_name,
        'status': status,
        'error': error,
        'num_images': result.get('num_images') if isinstance(result, dict) else None,
        'threshold': result.get('threshold') if isinstance(result, dict) else None,
        'accuracy': metrics.get('accuracy'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'f1': metrics.get('f1'),
        'auroc': metrics.get('auroc'),
        'pixel_auroc': metrics.get('pixel_auroc'),
        'selection_score': selection_score(metrics),
        'config_path': str(config_path),
        'checkpoint_path': str(checkpoint_path),
    }

    if model_name == 'autoencoder' and Path(checkpoint_path).exists():
        info = load_autoencoder_training_info(checkpoint_path)
        row.update({
            'ae_training_mode': info.get('training_mode'),
            'ae_final_epoch': info.get('final_epoch'),
            'ae_best_epoch': info.get('best_epoch'),
            'ae_best_val_loss': info.get('best_val_loss'),
            'ae_early_stopped': info.get('early_stopped'),
            'ae_elapsed_sec': info.get('elapsed_sec'),
            'ae_train_size': info.get('train_size'),
            'ae_val_size': info.get('val_size'),
            'ae_history_csv': info.get('history_csv'),
            'ae_history_png': info.get('history_png'),
            'ae_resume_enabled': info.get('resume_enabled'),
            'ae_resume_checkpoint': info.get('resume_checkpoint'),
            'ae_resumed_from_epoch': info.get('resumed_from_epoch'),
            'ae_save_every_epochs': info.get('save_every_epochs'),
        })
    return row


def save_results_csv(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    base_fieldnames = [
        'dataset', 'category', 'model', 'status', 'error', 'num_images', 'threshold',
        'accuracy', 'precision', 'recall', 'f1', 'auroc', 'pixel_auroc', 'selection_score',
        'config_path', 'checkpoint_path',
        'ae_training_mode', 'ae_final_epoch', 'ae_best_epoch', 'ae_best_val_loss',
        'ae_early_stopped', 'ae_elapsed_sec', 'ae_train_size', 'ae_val_size',
        'ae_history_csv', 'ae_history_png', 'ae_resume_enabled', 'ae_resume_checkpoint',
        'ae_resumed_from_epoch', 'ae_save_every_epochs',
    ]

    extra_fieldnames = []
    for row in rows:
        for key in row.keys():
            if key not in base_fieldnames and key not in extra_fieldnames:
                extra_fieldnames.append(key)

    fieldnames = base_fieldnames + extra_fieldnames
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

    print('saved:', path)
    return path


def load_existing_results_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    print(f'loaded existing results: {len(rows)} rows from {path}')
    return rows


def upsert_result_row(rows, new_row):
    key = (str(new_row.get('dataset')), str(new_row.get('category')), str(new_row.get('model')))
    replaced = False
    for idx, row in enumerate(rows):
        row_key = (str(row.get('dataset')), str(row.get('category')), str(row.get('model')))
        if row_key == key:
            rows[idx] = new_row
            replaced = True
            break
    if not replaced:
        rows.append(new_row)
    return rows


def find_existing_result_row(rows, category, model_name):
    for row in rows:
        if (
            str(row.get('dataset')) == 'mvtec'
            and str(row.get('category')) == str(category)
            and str(row.get('model')) == str(model_name)
        ):
            return row
    return None


def is_ok(value):
    return str(value).lower() in {'ok', 'success', 'done'}


def path_exists_from_row(row, key):
    value = row.get(key)
    if not value:
        return False
    try:
        return Path(str(value)).exists()
    except Exception:
        return False


def load_metrics_json(metrics_path):
    metrics_path = Path(metrics_path)
    if not metrics_path.exists():
        return None
    try:
        with metrics_path.open('r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as exc:
        print('[WARN] metrics json load failed:', metrics_path, repr(exc))
        return None


In [10]:
# ===== 전체 category × model 학습 및 평가 =====
# 재실행/런타임 끊김 대응:
# - all_results.csv에 status=ok이고 checkpoint가 존재하면 해당 category/model은 건너뜁니다.
# - 최종 checkpoint는 없지만 autoencoder_resume.pt가 있으면 train_autoencoder_regularized가 그 epoch 다음부터 이어서 학습합니다.
# - 최종 checkpoint는 있고 metrics json만 없으면 재학습하지 않고 평가만 다시 수행합니다.

all_results_path = RESULT_ROOT / 'all_results.csv'
rows = load_existing_results_csv(all_results_path)

for category in categories:
    print()
    print('=' * 80)
    print('CATEGORY:', category)
    print('=' * 80)

    # 데이터셋 확인
    try:
        train_set = build_dataset(make_config(category, 'autoencoder'), split='train')
        test_set = build_dataset(make_config(category, 'autoencoder'), split='test')
        print('train images:', len(train_set), 'test images:', len(test_set))
    except Exception as exc:
        print('[SKIP] dataset load failed:', repr(exc))
        for model_name in MODELS_TO_RUN:
            err_row = flatten_result(category, model_name, {}, '', '', status='dataset_error', error=repr(exc))
            rows = upsert_result_row(rows, err_row)
        save_results_csv(rows, all_results_path)
        continue

    for model_name in MODELS_TO_RUN:
        print()
        print('--- model:', model_name, '---')

        config = make_config(category, model_name)
        config_path = RESULT_ROOT / 'configs' / 'mvtec' / category / f'{model_name}.yaml'
        write_yaml(config, config_path)

        checkpoint_path = Path(config['outputs']['checkpoint'])
        resume_checkpoint_path = Path(config['outputs'].get('resume_checkpoint', str(checkpoint_path).replace('.pt', '_resume.pt')))
        metrics_path = RESULT_ROOT / 'metrics' / 'mvtec' / category / f'{model_name}.json'

        existing_row = find_existing_result_row(rows, category, model_name)

        # 1) 이미 완료된 row + checkpoint가 있으면 완전히 건너뜀
        if (
            SKIP_COMPLETED_RUNS
            and existing_row is not None
            and is_ok(existing_row.get('status'))
            and checkpoint_path.exists()
        ):
            print('[SKIP COMPLETED] 이미 완료된 결과가 있어 건너뜁니다.')
            print(' - checkpoint:', checkpoint_path)
            print(' - selection_score:', existing_row.get('selection_score'))
            continue

        # 2) row는 없거나 미완료지만, final checkpoint + metrics json이 있으면 row만 복구하고 건너뜀
        if SKIP_COMPLETED_RUNS and checkpoint_path.exists() and metrics_path.exists():
            restored_result = load_metrics_json(metrics_path)
            if restored_result is not None:
                restored_row = flatten_result(category, model_name, restored_result, config_path, checkpoint_path)
                rows = upsert_result_row(rows, restored_row)
                save_results_csv(rows, all_results_path)
                print('[RESTORED COMPLETED] checkpoint와 metrics json에서 결과 row를 복구하고 건너뜁니다.')
                continue

        try:
            force_retrain = (
                (model_name == 'autoencoder' and FORCE_RETRAIN_AUTOENCODER)
                or (model_name == 'patchcore' and FORCE_RETRAIN_PATCHCORE)
            )

            if checkpoint_path.exists() and SKIP_EXISTING_CHECKPOINTS and not force_retrain:
                print('checkpoint exists, skip train:', checkpoint_path)

            else:
                checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

                if checkpoint_path.exists() and force_retrain:
                    print('existing checkpoint will be overwritten:', checkpoint_path)

                if model_name == 'autoencoder':
                    if resume_checkpoint_path.exists() and AUTOENCODER_RESUME_ENABLED and not checkpoint_path.exists():
                        print('[RESUME READY] 임시 checkpoint에서 이어서 학습합니다:', resume_checkpoint_path)
                    elif resume_checkpoint_path.exists() and AUTOENCODER_RESUME_ENABLED and force_retrain:
                        print('[RESUME READY] force_retrain=True이지만 resume checkpoint가 있어 이어서 학습합니다:', resume_checkpoint_path)
                    else:
                        print('[TRAIN START] 새 학습 또는 기존 final checkpoint 없음')

                    train_autoencoder_regularized(config, device)

                elif model_name == 'patchcore':
                    train_patchcore(config, device)

                else:
                    raise ValueError(model_name)

            # checkpoint가 생성되었거나 기존 checkpoint가 있으면 평가 수행
            result = evaluate(config, checkpoint=checkpoint_path, split='test', output_path=metrics_path)
            row = flatten_result(category, model_name, result, config_path, checkpoint_path)
            print('metrics:', row)
            rows = upsert_result_row(rows, row)

        except Exception as exc:
            print('[ERROR]', category, model_name, repr(exc))
            err_row = flatten_result(category, model_name, {}, config_path, checkpoint_path, status='error', error=repr(exc))
            rows = upsert_result_row(rows, err_row)

        save_results_csv(rows, all_results_path)

print()
print('완료')
print('results:', all_results_path)


loaded existing results: 9 rows from /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/all_results.csv

CATEGORY: bottle
train images: 209 test images: 83

--- model: autoencoder ---
[TRAIN START] 새 학습 또는 기존 final checkpoint 없음
AutoEncoder detailed training
 - train samples: 178
 - validation samples: 31
 - max_epochs: 80 min_epochs: 20 patience: 4
 - resume_checkpoint: /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/bottle/autoencoder_resume.pt
 - resume: 새 학습 시작


epoch 1/80:   0%|          | 0/23 [00:00<?, ?it/s]

epoch=001 train_loss=1.462460 val_loss=0.808032 best_val=0.808032 lr=5.00e-04 no_improve=0/4 *


epoch 2/80:   0%|          | 0/23 [00:00<?, ?it/s]

epoch=002 train_loss=0.532603 val_loss=0.411046 best_val=0.411046 lr=5.00e-04 no_improve=0/4 *


...

saved_checkpoint=/content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/hazelnut/patchcore.pt
threshold_p95=0.83477974


metrics: {'dataset': 'mvtec', 'category': 'hazelnut', 'model': 'patchcore', 'status': 'ok', 'error': '', 'num_images': 110, 'threshold': 0.8347797393798828, 'accuracy': 0.8272727272727273, 'precision': 0.9636363636363636, 'recall': 0.7571428571428571, 'f1': 0.848, 'auroc': 0.9707142857142856, 'pixel_auroc': 0.9708056593448299, 'selection_score': 0.9257144018491945, 'config_path': '/content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/configs/mvtec/hazelnut/patchcore.yaml', 'checkpoint_path': '/content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/hazelnut/patchcore.pt'}
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/all_results.csv

CATEGORY: leather
train images: 245 test images: 124

--- model: autoencoder ---
[TRAIN START] 새 학습 또는 기존 final checkpoint 없음
AutoEncoder detailed training
 - train samples: 208
 - validation samples: 37
 - max_epochs: 80 min_epochs: 20 patience: 4
 - resume_checkpoint: /content/drive/MyDrive/DefectVis

epoch 1/80:   0%|          | 0/26 [00:00<?, ?it/s]

[INTERRUPTED] 사용자가 실행을 중단했습니다. 마지막 완료 epoch 기준 resume checkpoint를 저장합니다.


KeyboardInterrupt: 

In [11]:
# ===== 결과 테이블, best model 계산 =====
import pandas as pd

all_results_path = RESULT_ROOT / 'all_results.csv'
results_df = pd.read_csv(all_results_path)

ok_df = results_df[results_df['status'] == 'ok'].copy()
for col in [
    'accuracy', 'precision', 'recall', 'f1', 'auroc', 'pixel_auroc', 'selection_score',
    'ae_final_epoch', 'ae_best_epoch', 'ae_best_val_loss', 'ae_elapsed_sec',
]:
    if col in ok_df.columns:
        ok_df[col] = pd.to_numeric(ok_df[col], errors='coerce')

leaderboard = ok_df.sort_values(['selection_score', 'auroc', 'f1', 'accuracy'], ascending=False)
best_by_category = leaderboard.groupby(['dataset', 'category'], as_index=False).first()
model_summary = ok_df.groupby('model', as_index=False)[['accuracy', 'precision', 'recall', 'f1', 'auroc', 'pixel_auroc', 'selection_score']].mean(numeric_only=True)
category_summary = ok_df.groupby('category', as_index=False)[['accuracy', 'f1', 'auroc', 'selection_score']].mean(numeric_only=True)

autoencoder_history_summary = ok_df[ok_df['model'] == 'autoencoder'].copy()
if not autoencoder_history_summary.empty:
    keep_cols = [c for c in [
        'category', 'model', 'accuracy', 'f1', 'auroc', 'selection_score',
        'ae_final_epoch', 'ae_best_epoch', 'ae_best_val_loss', 'ae_early_stopped', 'ae_history_png'
    ] if c in autoencoder_history_summary.columns]
    autoencoder_history_summary = autoencoder_history_summary[keep_cols].sort_values('category')
    autoencoder_history_summary.to_csv(RESULT_ROOT / 'autoencoder_history_summary.csv', index=False)

leaderboard.to_csv(RESULT_ROOT / 'leaderboard.csv', index=False)
best_by_category.to_csv(RESULT_ROOT / 'best_by_category.csv', index=False)
model_summary.to_csv(RESULT_ROOT / 'model_summary.csv', index=False)
category_summary.to_csv(RESULT_ROOT / 'category_summary.csv', index=False)

print('best_by_category')
display(best_by_category)
print('model_summary')
display(model_summary)
if not autoencoder_history_summary.empty:
    print('autoencoder_history_summary')
    display(autoencoder_history_summary)

best_by_category


,dataset,category,model,status,error,num_images,threshold,accuracy,precision,recall,...,ae_early_stopped,ae_elapsed_sec,ae_train_size,ae_val_size,ae_history_csv,ae_history_png,ae_resume_enabled,ae_resume_checkpoint,ae_resumed_from_epoch,ae_save_every_epochs
0,mvtec,bottle,patchcore,ok,NaN,83,0.721049,0.951807,0.940299,1.000000,...,True,1888.547811,178.0,31.0,/content/drive/MyDrive/DefectVision-AD/outputs...,/content/drive/MyDrive/DefectVision-AD/outputs...,True,/content/drive/MyDrive/DefectVision-AD/outputs...,0.0,10.0
1,mvtec,cable,patchcore,ok,NaN,150,0.807863,0.833333,0.924051,0.793478,...,True,4261.673045,190.0,34.0,/content/drive/MyDrive/DefectVision-AD/outputs...,/content/drive/MyDrive/DefectVision-AD/outputs...,True,/content/drive/MyDrive/DefectVision-AD/outputs...,0.0,10.0
2,mvtec,capsule,patchcore,ok,NaN,132,0.697209,0.787879,0.987952,0.752294,...,True,1470.432342,186.0,33.0,/content/drive/MyDrive/DefectVision-AD/outputs...,/content/drive/MyDrive/DefectVision-AD/outputs...,True,/content/drive/MyDrive/DefectVision-AD/outputs...,0.0,10.0
3,mvtec,carpet,patchcore,ok,NaN,117,0.680991,0.923077,0.916667,0.988764,...,True,3806.542553,238.0,42.0,/content/drive/MyDrive/DefectVision-AD/outputs...,/content/drive/MyDrive/DefectVision-AD/outputs...,True,/content/drive/MyDrive/DefectVision-AD/outputs...,0.0,10.0
4,mvtec,grid,patchcore,ok,NaN,78,0.578788,0.923077,0.932203,0.964912,...,True,3268.183587,224.0,40.0,/content/drive/MyDrive/DefectVision-AD/outputs...,/content/drive/MyDrive/DefectVision-AD/outputs...,True,/content/drive/MyDrive/DefectVision-AD/outputs...,0.0,10.0
5,mvtec,hazelnut,patchcore,ok,NaN,110,0.834780,0.827273,0.963636,0.757143,...,True,3166.150972,332.0,59.0,/content/drive/MyDrive/DefectVision-AD/outputs...,/content/drive/MyDrive/DefectVision-AD/outputs...,True,/content/drive/MyDrive/DefectVision-AD/outputs...,0.0,10.0


model_summary


,model,accuracy,precision,recall,f1,auroc,pixel_auroc,selection_score
0,autoencoder,0.431539,0.821732,0.251111,0.368686,0.599067,0.701944,0.550438
1,patchcore,0.874408,0.944135,0.876099,0.904138,0.967332,0.959820,0.940363


autoencoder_history_summary


,category,model,accuracy,f1,auroc,selection_score,ae_final_epoch,ae_best_epoch,ae_best_val_loss,ae_early_stopped,ae_history_png
9,bottle,autoencoder,0.554217,0.602151,0.652381,0.645164,34.0,30.0,0.007447,True,/content/drive/MyDrive/DefectVision-AD/outputs...
1,cable,autoencoder,0.393333,0.061856,0.452024,0.363814,66.0,62.0,0.018879,True,/content/drive/MyDrive/DefectVision-AD/outputs...
3,capsule,autoencoder,0.250000,0.208000,0.507379,0.473556,23.0,19.0,0.005311,True,/content/drive/MyDrive/DefectVision-AD/outputs...
5,carpet,autoencoder,0.324786,0.357724,0.374799,0.416524,48.0,44.0,0.019462,True,/content/drive/MyDrive/DefectVision-AD/outputs...
7,grid,autoencoder,0.448718,0.410959,0.746032,0.622360,49.0,45.0,0.012879,True,/content/drive/MyDrive/DefectVision-AD/outputs...
10,hazelnut,autoencoder,0.618182,0.571429,0.861786,0.781212,28.0,24.0,0.003856,True,/content/drive/MyDrive/DefectVision-AD/outputs...


In [12]:
# ===== README 및 Streamlit용 시각화 PNG 저장 =====
import matplotlib.pyplot as plt

FIG_DIR = RESULT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_bar(df, x, y, title, filename, xlabel=None, ylabel=None, horizontal=False):
    if df.empty:
        print('skip empty:', filename)
        return None
    fig, ax = plt.subplots(figsize=(11, max(5, 0.35 * len(df)) if horizontal else 6))
    if horizontal:
        ax.barh(df[x].astype(str), df[y])
        ax.invert_yaxis()
    else:
        ax.bar(df[x].astype(str), df[y])
        ax.tick_params(axis='x', rotation=45)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    ax.grid(axis='y' if not horizontal else 'x', alpha=0.3)
    fig.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, dpi=160)
    plt.close(fig)
    print('saved:', path)
    return path

# 1) 모델 평균 selection score
save_bar(
    model_summary.sort_values('selection_score', ascending=False),
    'model', 'selection_score',
    'Average model selection score',
    'fig_model_average_selection_score.png',
    xlabel='Model', ylabel='Selection score'
)

# 2) category별 평균 accuracy
save_bar(
    category_summary.sort_values('accuracy', ascending=False),
    'category', 'accuracy',
    'Average accuracy by MVTec category',
    'fig_category_average_accuracy.png',
    xlabel='Category', ylabel='Accuracy', horizontal=True
)

# 3) category별 best model score
best_plot = best_by_category.copy()
best_plot['category_model'] = best_plot['category'] + ' / ' + best_plot['model']
save_bar(
    best_plot.sort_values('selection_score', ascending=False),
    'category_model', 'selection_score',
    'Best model by category',
    'fig_best_model_by_category.png',
    xlabel='Category / model', ylabel='Selection score', horizontal=True
)

# 4) model x category heatmap: selection score
pivot = ok_df.pivot_table(index='category', columns='model', values='selection_score', aggfunc='mean')
if not pivot.empty:
    fig, ax = plt.subplots(figsize=(8, max(5, 0.4 * len(pivot))))
    im = ax.imshow(pivot.fillna(0).values, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title('Selection score heatmap by category and model')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            value = pivot.iloc[i, j]
            text = '-' if pd.isna(value) else f'{value:.3f}'
            ax.text(j, i, text, ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, label='Selection score')
    fig.tight_layout()
    heatmap_path = FIG_DIR / 'fig_selection_score_heatmap.png'
    fig.savefig(heatmap_path, dpi=160)
    plt.close(fig)
    print('saved:', heatmap_path)

# 5) category × model accuracy grouped style using line markers
if not ok_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    for model_name, sub in ok_df.sort_values('category').groupby('model'):
        ax.plot(sub['category'], sub['accuracy'], marker='o', label=model_name)
    ax.set_title('Accuracy by category and model')
    ax.set_xlabel('Category')
    ax.set_ylabel('Accuracy')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    path = FIG_DIR / 'fig_accuracy_by_category_model.png'
    fig.savefig(path, dpi=160)
    plt.close(fig)
    print('saved:', path)

# 6) AutoEncoder validation loss by category
if 'ae_best_val_loss' in ok_df.columns:
    ae_loss_df = ok_df[(ok_df['model'] == 'autoencoder') & ok_df['ae_best_val_loss'].notna()].copy()
    if not ae_loss_df.empty:
        save_bar(
            ae_loss_df.sort_values('ae_best_val_loss'),
            'category', 'ae_best_val_loss',
            'AutoEncoder best validation loss by category',
            'fig_autoencoder_best_val_loss_by_category.png',
            xlabel='Category', ylabel='Best validation loss', horizontal=True
        )

# 7) AutoEncoder best epoch by category
if 'ae_best_epoch' in ok_df.columns:
    ae_epoch_df = ok_df[(ok_df['model'] == 'autoencoder') & ok_df['ae_best_epoch'].notna()].copy()
    if not ae_epoch_df.empty:
        save_bar(
            ae_epoch_df.sort_values('ae_best_epoch', ascending=False),
            'category', 'ae_best_epoch',
            'AutoEncoder best epoch by category',
            'fig_autoencoder_best_epoch_by_category.png',
            xlabel='Category', ylabel='Best epoch', horizontal=True
        )

saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/figures/fig_model_average_selection_score.png
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/figures/fig_category_average_accuracy.png
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/figures/fig_best_model_by_category.png
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/figures/fig_selection_score_heatmap.png
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/figures/fig_accuracy_by_category_model.png
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/figures/fig_autoencoder_best_val_loss_by_category.png
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/figures/fig_autoencoder_best_epoch_by_category.png


In [13]:
# ===== Streamlit demo용 test sample pool 생성 =====
# 학습에 쓰이지 않은 test 이미지에서 category별 sample pool을 만듭니다.
# Streamlit 앱은 여기서 매번 10개를 랜덤하게 보여줍니다.

from PIL import Image
import random
import hashlib

SAMPLE_ROOT = DEPLOY_ROOT / 'demo_samples' / 'mvtec'
SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)
metadata = []

rng = random.Random(42)
for category in categories:
    config = make_config(category, 'autoencoder')
    try:
        test_set = build_dataset(config, split='test')
    except Exception as exc:
        print('sample skip:', category, repr(exc))
        continue

    items = list(test_set.items)
    rng.shuffle(items)
    selected = items[:min(SAMPLE_POOL_PER_CATEGORY, len(items))]
    out_dir = SAMPLE_ROOT / category
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for idx, item in enumerate(selected):
        src = Path(item.image_path)
        safe_stem = hashlib.sha1(str(src).encode('utf-8')).hexdigest()[:10]
        out_name = f'{idx:03d}_{item.defect_type}_{safe_stem}.jpg'
        out_path = out_dir / out_name
        img = Image.open(src).convert('RGB')
        img.thumbnail((512, 512))
        img.save(out_path, quality=90)
        metadata.append({
            'dataset': 'mvtec',
            'category': category,
            'sample_path': str(out_path.relative_to(DEPLOY_ROOT)),
            'original_path': str(src),
            'defect_type': item.defect_type,
            'label': int(item.label),
            'label_name': 'anomaly' if int(item.label) else 'normal',
        })

sample_meta_path = DEPLOY_ROOT / 'demo_samples_manifest.json'
sample_meta_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print('saved:', sample_meta_path)
print('num samples:', len(metadata))

saved: /content/drive/MyDrive/DefectVision-AD/deploy/demo_samples_manifest.json
num samples: 450


In [14]:
# ===== best model registry 생성 =====
# registry에는 checkpoint의 원래 Drive 경로와 EC2 배포 시 상대 경로를 모두 저장합니다.

registry = {
    'project': 'DefectVision-AD',
    'dataset': 'mvtec',
    'image_size': IMAGE_SIZE,
    'selection_weights': SELECTION_WEIGHTS,
    'models': [],
}

for _, row in best_by_category.iterrows():
    checkpoint_path = Path(str(row['checkpoint_path']))
    registry['models'].append({
        'dataset': 'mvtec',
        'category': str(row['category']),
        'model': str(row['model']),
        'checkpoint_drive_path': str(checkpoint_path),
        'checkpoint_relative_path': f"models/mvtec/{row['category']}/{row['model']}.pt",
        'accuracy': optional_float(row.get('accuracy')),
        'precision': optional_float(row.get('precision')),
        'recall': optional_float(row.get('recall')),
        'f1': optional_float(row.get('f1')),
        'auroc': optional_float(row.get('auroc')),
        'pixel_auroc': optional_float(row.get('pixel_auroc')),
        'selection_score': optional_float(row.get('selection_score')),
        'threshold': optional_float(row.get('threshold')),
    })

registry_path = DEPLOY_ROOT / 'model_registry.json'
registry_path.write_text(json.dumps(registry, indent=2, ensure_ascii=False), encoding='utf-8')
print('saved:', registry_path)
print(json.dumps(registry, indent=2, ensure_ascii=False)[:2000])

saved: /content/drive/MyDrive/DefectVision-AD/deploy/model_registry.json
{
  "project": "DefectVision-AD",
  "dataset": "mvtec",
  "image_size": 256,
  "selection_weights": {
    "auroc": 0.4,
    "f1": 0.25,
    "pixel_auroc": 0.25,
    "accuracy": 0.1
  },
  "models": [
    {
      "dataset": "mvtec",
      "category": "bottle",
      "model": "patchcore",
      "checkpoint_drive_path": "/content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/bottle/patchcore.pt",
      "checkpoint_relative_path": "models/mvtec/bottle/patchcore.pt",
      "accuracy": 0.9518072289156626,
      "precision": 0.9402985074626866,
      "recall": 1.0,
      "f1": 0.9692307692307692,
      "auroc": 1.0,
      "pixel_auroc": 0.9778147642832916,
      "selection_score": 0.9819421062700816,
      "threshold": 0.7210485219955445
    },
    {
      "dataset": "mvtec",
      "category": "cable",
      "model": "patchcore",
      "checkpoint_drive_path": "/content/drive/MyDrive/DefectVision-AD/outputs/che